In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Customer Success Crew

## What We're Building

A four-agent customer success workflow focused on retention and response quality:

| Agent | Role |
|-------|------|
| Complaint Analyst | Diagnose complaint themes, severity, and sentiment |
| Churn Risk Agent | Estimate churn risk and identify leading indicators |
| Resolution Drafter | Draft response and recovery plan for the customer |
| Account Summarizer | Consolidate findings into an account health brief |

## Collaboration Flow

1. Complaint Analyst extracts issue patterns and escalation signals.
2. Churn Risk Agent uses complaint evidence to score retention risk.
3. Resolution Drafter uses complaint + churn context to craft practical recovery actions.
4. Account Summarizer merges all outputs into one clear account action brief.

This collaboration chain ensures each specialist contributes a distinct decision layer.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Customer Support Data

Simulated support ticket history for one customer account.

In [3]:
account_info = """
ACCOUNT: Acme Corp (Enterprise Plan — $2,400/mo)
- Customer since: 18 months
- Contract renewal: 45 days away
- Monthly usage: Declining (was 85% of quota, now 62%)
- Champion contact: Sarah Chen (VP Ops) — on PTO for 2 weeks
- Technical contact: James Park (Senior Engineer) — increasingly frustrated
"""

support_tickets = """
RECENT SUPPORT TICKETS (last 30 days):

TICKET #4521 (5 days ago) — Priority: HIGH
From: James Park
Subject: API timeouts during peak hours — THIRD TIME this month
Body: "Our dashboard keeps timing out between 9-11am EST. This is the third
ticket I've filed about this. Last time I was told it was 'fixed' but clearly
it wasn't. We're evaluating alternatives. Please escalate."

TICKET #4490 (12 days ago) — Priority: MEDIUM
From: James Park
Subject: Missing data in export — rows dropped silently
Body: "Exported our Q3 report and noticed 2,300 rows are missing compared to
the in-app view. No error message. We had to manually reconcile. This caused
a 4-hour delay in our board presentation."

TICKET #4455 (20 days ago) — Priority: LOW
From: Lisa Wong (Analyst)
Subject: Feature request — custom date ranges in reports
Body: "Can we get custom date range filters? Right now it's only last 7/30/90
days. We need fiscal quarter views. Thanks!"

TICKET #4401 (28 days ago) — Priority: HIGH
From: James Park
Subject: API timeouts again
Body: "Same issue as ticket #4350. API returns 504 during morning hours.
Attached logs. This is blocking our automated reporting pipeline."
"""

csat_scores = """
CSAT SURVEY RESPONSES:
- 6 months ago: 9/10 ("Love the product, works great")
- 3 months ago: 6/10 ("Good product but reliability issues")
- 1 month ago: 3/10 ("Frustrated. Considering switching.")
"""

print(f"Loaded: account info, {support_tickets.count('TICKET')} tickets, 3 CSAT scores")

Loaded: account info, 5 tickets, 3 CSAT scores


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    complaint_analyst = Agent(
        role="Complaint Analyst",
        goal="Parse customer complaints and classify severity, urgency, and themes",
        backstory=(
            "You are a customer support quality analyst who detects escalation signals"
            " and recurring friction points from ticket narratives."
        ),
        llm=llm,
        verbose=True,
    )

    churn_risk_agent = Agent(
        role="Churn Risk Agent",
        goal="Assess churn probability and identify top risk/protective factors",
        backstory=(
            "You are a retention analyst who translates complaint patterns and behavior"
            " signals into churn risk scoring and intervention priorities."
        ),
        llm=llm,
        verbose=True,
    )

    resolution_drafter = Agent(
        role="Resolution Drafter",
        goal="Draft clear customer communication and a realistic resolution plan",
        backstory=(
            "You are a senior customer success manager skilled in de-escalation and"
            " trust recovery through concrete action plans."
        ),
        llm=llm,
        verbose=True,
    )

    account_summarizer = Agent(
        role="Account Summarizer",
        goal="Produce a concise account health summary with next-step actions",
        backstory=(
            "You synthesize cross-functional CS findings into one brief that leaders and"
            " account owners can execute immediately."
        ),
        llm=llm,
        verbose=True,
    )
else:
    complaint_analyst = {"role": "Complaint Analyst"}
    churn_risk_agent = {"role": "Churn Risk Agent"}
    resolution_drafter = {"role": "Resolution Drafter"}
    account_summarizer = {"role": "Account Summarizer"}

print("4 agents created: Complaint Analyst, Churn Risk Agent, Resolution Drafter, Account Summarizer")


4 agents created: Complaint Analyst, Churn Risk Agent, Resolution Drafter, Account Summarizer


## Step 4 — Create Tasks

Notice: the final task uses `output_file` — CrewAI will automatically save
the task output to a file on disk.

In [5]:
if CREWAI_AVAILABLE:
    complaint_task = Task(
        description=f"""Analyze the customer complaint history and extract:
1. Main complaint themes
2. Severity/urgency classification
3. Sentiment trend
4. Escalation triggers
5. Evidence quotes

Use this source data:
{customer_data}""",
        expected_output="Complaint analysis with ranked issue themes and escalation signals.",
        agent=complaint_analyst,
    )

    churn_task = Task(
        description="""Based on complaint analysis, provide:
1. Churn score (0-100)
2. Top risk factors
3. Protective factors
4. Likely tipping point
5. Intervention priority""",
        expected_output="Churn risk assessment with rationale and intervention priority.",
        agent=churn_risk_agent,
        context=[complaint_task],
    )

    resolution_task = Task(
        description="""Draft:
1. Customer response message
2. Resolution timeline with milestones
3. Ownership and follow-up cadence
4. Recovery offer/plan if applicable
5. Internal handoff notes""",
        expected_output="Resolution response and recovery action plan.",
        agent=resolution_drafter,
        context=[complaint_task, churn_task],
    )

    account_summary_task = Task(
        description="""Produce an account summary brief:
1. Complaint snapshot
2. Churn risk snapshot
3. Resolution plan snapshot
4. Next 30-day retention actions
5. Account owner checklist

Keep it concise and actionable.""",
        expected_output="Consolidated account health and action summary.",
        agent=account_summarizer,
        context=[complaint_task, churn_task, resolution_task],
        markdown=True,
    )
else:
    complaint_task = {"name": "Complaint analysis task"}
    churn_task = {"name": "Churn risk task"}
    resolution_task = {"name": "Resolution drafting task"}
    account_summary_task = {"name": "Account summary task"}

print("4 tasks created with collaboration flow: complaint -> churn -> resolution -> account summary")


4 tasks created with collaboration flow: complaint -> churn -> resolution -> account summary


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Account summary generated: high churn risk detected, escalation response drafted, and 30-day retention plan created."
        self.tasks_output = [
            _DemoTaskOutput("Complaint Analysis", "Recurring onboarding + reliability complaints with negative sentiment acceleration."),
            _DemoTaskOutput("Churn Risk", "Churn risk score elevated; key trigger is unresolved SLA concerns."),
            _DemoTaskOutput("Resolution Draft", "Customer response and milestone-based remediation plan drafted."),
            _DemoTaskOutput("Account Summary", "One-page account brief with ownership and next actions."),
        ]


if CREWAI_AVAILABLE:
    cs_crew = Crew(
        agents=[complaint_analyst, churn_risk_agent, resolution_drafter, account_summarizer],
        tasks=[complaint_task, churn_task, resolution_task, account_summary_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Customer Success crew assembled.")
    result = cs_crew.kickoff()
    print("\nAccount summary complete.")
else:
    print("Demo mode: showing collaboration flow without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing collaboration flow without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("ACCOUNT SUMMARY BRIEF")
print("=" * 60)
print(result.raw)

preview("Complaint Analysis", result.tasks_output[0].raw)
preview("Churn Risk", result.tasks_output[1].raw)
preview("Resolution Draft", result.tasks_output[2].raw)
preview("Account Summary", result.tasks_output[3].raw)


ACCOUNT SUMMARY BRIEF
Account summary generated: high churn risk detected, escalation response drafted, and 30-day retention plan created.

Complaint Analysis
Recurring onboarding + reliability complaints with negative sentiment acceleration.

Churn Risk
Churn risk score elevated; key trigger is unresolved SLA concerns.

Resolution Draft
Customer response and milestone-based remediation plan drafted.

Account Summary
One-page account brief with ownership and next actions.


In [8]:
# View individual task outputs
labels = ["Complaint Analysis", "Churn Risk Assessment", "Resolution Drafts", "Health Report"]
for label, task_out in zip(labels, result.tasks_output):
    print(f"\n{'=' * 60}")
    print(f"📌 {label}")
    print("=" * 60)
    print(task_out.raw[:800])
    if len(task_out.raw) > 800:
        print("... (truncated)")


📌 Complaint Analysis
Recurring onboarding + reliability complaints with negative sentiment acceleration.

📌 Churn Risk Assessment
Churn risk score elevated; key trigger is unresolved SLA concerns.

📌 Resolution Drafts
Customer response and milestone-based remediation plan drafted.

📌 Health Report
One-page account brief with ownership and next actions.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Agent specialization | Complaint diagnosis, churn modeling, response drafting, account synthesis |
| Collaboration flow | Sequential handoff across analysis, risk, remediation, and summary |
| Explicit context wiring | Downstream tasks consume upstream outputs via `context=[...]` |
| CS execution focus | Final brief ends with concrete retention actions and owners |

## Extensions

- Integrate CRM events and ticket metadata for richer churn signals
- Add SLA breach detector as a pre-churn trigger task
- Export account summary to CSM playbook templates
- Add week-over-week churn risk trend tracking
